In [6]:
import openpyxl as pxl
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("./starter-kit/data/numeros_tva.csv") 

In [7]:
#df.head(3)
#df.info()
df.columns


Index(['id', 'raison_sociale', 'pays_declare', 'numero_tva', 'date_saisie',
       'source_saisie'],
      dtype='str')

In [8]:
#de pays distincts 
df["pays_declare"].nunique()

15

In [9]:
# Combien de formats différents pour un même pays ?
df.groupby("pays_declare")["numero_tva"].nunique()

pays_declare
BE    919
DK    918
FI    866
FR    933
GB    103
IT    909
LU    927
NL    909
PL    911
PT    906
QQ    108
SE    911
UK    103
XX     87
ZZ    115
Name: numero_tva, dtype: int64

In [10]:
#2-Quelle proportion de lignes porte des caractères qui n'ont rien à faire dans un numéro de TVA ?
nb_lignes = len(df)

lignes_anormales = df["numero_tva"].astype(str).str.contains(
    r"[^A-Za-z0-9]",
    regex=True,
    na=False
).sum()

print("Nombre de lignes avec caractères anormaux :", lignes_anormales)


Nombre de lignes avec caractères anormaux : 1813


In [22]:
# combien de valeur vide 
vide = df["numero_tva"].isna()
print("",vide.sum())
#combien de formes différentes
valeur = ["-", " ", "", "null", "N/A"]
for v in valeur:
    print(v, ":", (df["numero_tva"] == v).sum())
print("Nombre de lignes avec caractères anormaux :",len(df[df["numero_tva"].isin(valeur)]))    

 146
- : 55
  : 59
 : 0
null : 0
N/A : 0
Nombre de lignes avec caractères anormaux : 114


In [12]:
df["correspondance"] = df["pays_declare"] == df["numero_tva"].str[:2]
df["correspondance"].value_counts()

correspondance
True     8355
False    1645
Name: count, dtype: int64

In [13]:
#Le CSV et l'Excel contiennent-ils vraiment la même chose une fois chargés ? Lequel prenez-vous comme source de vérité, et pourquoi ?
df_tva= pd.read_excel("./starter-kit/data/numeros_tva.xlsx") 


In [14]:
df.equals(df_tva)
#le CSV et le fichier Excel contiennent exactement les mêmes données
#J'ai choisi le fichier CSV comme source de vérité, car il contient les mêmes données que le fichier Excel et il est facile à exploiter avec Python et Pandas.

False

In [15]:
#Les canaux de saisie et les périodes de saisie se ressemblent-ils, ou certains sortent-ils du lot ?
df["source_saisie"].value_counts()

source_saisie
reprise_erp           2071
crm                   2019
portail_client        1990
saisie_manuelle       1976
import_fournisseur    1944
Name: count, dtype: int64

In [16]:
df["date_saisie"] = pd.to_datetime(df["date_saisie"])
df["date_saisie"].dt.year.value_counts().sort_index()

date_saisie
2024    5760
2025    4240
Name: count, dtype: int64

In [17]:
# Observation et conclusion 
# Après avoir exploré le fichier, j'ai remarqué plusieurs problèmes : 
# plusieurs numéros de TVA sont vides ou contiennent des formes différentes, les numéros de TVA d'un même pays ne sont pas toujours écrits de la même manière.
# certains pays déclarés ne correspondent pas au préfixe des numéros de TVA.
# les fichiers CSV et Excel contiennent les mêmes données. J'ai choisi le CSV comme source de vérité. 